In [1]:
import pandas as pd

# Load the CSV file
file_path = r"/Users/saurabhlevin/Deployment/IDS-DRR-Himachal-Pradesh-Risk-Score-Model/RiskScoreModel/data/risk_score_final_district.csv"
print("Loading CSV file from:", file_path)
df = pd.read_csv(file_path)
print("CSV file loaded successfully. Shape:", df.shape)

Loading CSV file from: /Users/saurabhlevin/Deployment/IDS-DRR-Himachal-Pradesh-Risk-Score-Model/RiskScoreModel/data/risk_score_final_district.csv
CSV file loaded successfully. Shape: (6324, 86)


In [1]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/saurabhlevin/Deployment/IDS-DRR-Himachal-Pradesh-Risk-Score-Model/RiskScoreModel/scripts/tests_deployment


In [2]:
df.columns


Index(['object-id', 'district', 'timeperiod', 'total-tender-awarded-value',
       'repair-and-restoration-tenders-awarded-value',
       'lwss-tenders-awarded-value', 'ndrf-tenders-awarded-value',
       'sdmf-tenders-awarded-value', 'wss-tenders-awarded-value',
       'restoration-measures-tenders-awarded-value',
       'immediate-measures-tenders-awarded-value',
       'others-tenders-awarded-value', 'state', 'max-rain', 'mean-rain',
       'count', 'sum-rain', 'shape-leng', 'shape-area', 'tehsil',
       'mean-daily-runoff', 'sum-runoff', 'peak-runoff', 'inundation-pct',
       'inundation-intensity-mean', 'inundation-intensity-mean-nonzero',
       'inundation-intensity-sum', 'structure-lost', 'health-centres-lost',
       'health-amount', 'internalwatersupply', 'electricwires',
       'electricpoles', 'roadlength', 'streetlights', 'person-dead',
       'person-major-injury', 'schools-damaged', 'economic-loss',
       'total-livestock-loss', 'relief-and-mitigation-sanction-value',

In [4]:
# Selecting only numeric columns along with 'object-id', 'timeperiod', and 'financial-year'
numeric_columns = df.select_dtypes(include=["number"]).columns
list(numeric_columns).remove("shape-area")
numeric_columns = numeric_columns.drop("shape-area")
df_numeric = df[["object-id", "timeperiod", "financial-year"] + list(numeric_columns)]

# Identifying columns that are not included
excluded_columns = [col for col in df.columns if col not in df_numeric.columns]
print("Excluded columns:", excluded_columns)

df_numeric.head()

Excluded columns: ['district', 'state', 'shape-area', 'tehsil']


,object-id,timeperiod,financial-year,total-tender-awarded-value,repair-and-restoration-tenders-awarded-value,lwss-tenders-awarded-value,ndrf-tenders-awarded-value,sdmf-tenders-awarded-value,wss-tenders-awarded-value,restoration-measures-tenders-awarded-value,...,sdmf-tenders-awarded-value-fy-cumsum,wss-tenders-awarded-value-fy-cumsum,preparedness-measures-tenders-awarded-value-fy-cumsum,immediate-measures-tenders-awarded-value-fy-cumsum,others-tenders-awarded-value-fy-cumsum,relief-and-mitigation-sanction-value-fy-cumsum,topsis-score,risk-score,Unnamed: 0,total-infrastructure-damage
0,02-032-00052,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.804188,5,NaN,0.0
1,02-026-00014,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.804188,5,NaN,0.0
2,02-024-02130,2021_04,2021-2022,700000.0,700000.0,700000.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.804188,5,NaN,0.0
3,02-024-02127,2021_04,2021-2022,12367089.0,11517343.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,849746.0,0.0,0.797100,5,NaN,0.0
4,02-032-00056,2021_04,2021-2022,1161555.0,1161555.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.786408,5,NaN,0.0


In [5]:

df_melted = df_numeric.melt(id_vars=["object-id", "timeperiod", "financial-year"], var_name="factor", value_name="score")
print("Data melted. Shape:", df_melted.shape)

df_melted.head()

Data melted. Shape: (499596, 5)


,object-id,timeperiod,financial-year,factor,score
0,02-032-00052,2021_04,2021-2022,total-tender-awarded-value,0.0
1,02-026-00014,2021_04,2021-2022,total-tender-awarded-value,0.0
2,02-024-02130,2021_04,2021-2022,total-tender-awarded-value,700000.0
3,02-024-02127,2021_04,2021-2022,total-tender-awarded-value,12367089.0
4,02-032-00056,2021_04,2021-2022,total-tender-awarded-value,1161555.0


In [6]:

df_transposed = df_melted.pivot(index=["factor", "timeperiod", "financial-year"], columns="object-id", values="score").reset_index()
print("Data pivoted. Shape:", df_transposed.shape)
df_transposed.head()
# Save or display the transformed data
output_file = r"/Users/saurabhlevin/Deployment/IDS-DRR-Himachal-Pradesh-Risk-Score-Model/RiskScoreModel/data/Transformed_HP_Data.csv"
df_transposed.to_csv(output_file, index=False)
print("Transformed data saved to:", output_file)

Data pivoted. Shape: (4029, 127)
Transformed data saved to: /Users/saurabhlevin/Deployment/IDS-DRR-Himachal-Pradesh-Risk-Score-Model/RiskScoreModel/data/Transformed_HP_Data.csv


In [7]:
# Verify with the source data. Given factor, district, timeperiod and object-id, the score should match

factor = "sdrf-sanctions-awarded-value"
district = '02-031-00202'
timeperiod = '2024_07'
df_transposed.head()
print("modified")
print(df_transposed[(df_transposed["factor"] == factor) & (df_transposed["timeperiod"] == timeperiod) ][[district, "timeperiod", "financial-year", "factor"]])
print("original")
df[(df["object-id"] == district) & (df["timeperiod"] == timeperiod)][['timeperiod', 'financial-year', factor]]



modified


KeyError: "['02-031-00202'] not in index"

In [8]:
dff = pd.read_csv("Transformed_Assam_Data.csv")
conditions = []
operator_map = {
            '==': lambda col, val: col == val,
            '!=': lambda col, val: col != val,
            '>': lambda col, val: col > val,
            '<': lambda col, val: col < val,
            '>=': lambda col, val: col >= val,
            '<=': lambda col, val: col <= val,
            'in': lambda col, val: col.isin(val),
            'not in': lambda col, val: ~col.isin(val)
        }
conditions.append(operator_map["=="](dff["factor"], "sdrf-sanctions-awarded-value"))

dff = dff[pd.concat(conditions, axis=1).all(axis=1)]
dff

,factor,timeperiod,financial-year,18-300,18-300-00101,18-300-00102,18-300-00103,18-300-00104,18-300-00105,18-300-00106,...,18-799,18-799-00124,18-799-00125,18-799-00265,18-816,18-816-00235,18-816-00236,18-816-00258,18-816-00259,18-816-00261
2835,sdrf-sanctions-awarded-value,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2836,sdrf-sanctions-awarded-value,2021_05,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2837,sdrf-sanctions-awarded-value,2021_06,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2838,sdrf-sanctions-awarded-value,2021_07,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2839,sdrf-sanctions-awarded-value,2021_08,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2840,sdrf-sanctions-awarded-value,2021_09,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2841,sdrf-sanctions-awarded-value,2021_10,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2842,sdrf-sanctions-awarded-value,2021_11,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2843,sdrf-sanctions-awarded-value,2021_12,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2844,sdrf-sanctions-awarded-value,2022_01,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
